# 04 - Ingesta Bronze: Catálogo CIIU (Actividades Económicas)

Trabajo Práctico 1 - Sección 3 (Fuente 3, complementaria): Pipeline de ingesta PySpark

## Fuente

* **Dataset**: Catálogo de actividades económicas (códigos CIIU)
* **API**: `https://www.datos.gov.co/resource/nuke-fusu.json` (Socrata / SoQL)
* **Volumen**: 499 registros (catálogo estático, no crece con el tiempo)
* **Destino**: `Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api`

## Por qué esta fuente complementa a RUES

RUES (`02_Ingesta_Bronze_RUES.ipynb`) trae columnas como `cod_ciiu_act_econ_pri`, `cod_ciiu_act_econ_sec`, `ciiu3` y `ciiu4`, pero **solo el código** (ej. `"2211"`), sin ninguna descripción de a qué actividad económica corresponde. Este catálogo es el maestro código→descripción (columna `code` con el mismo formato de 4 dígitos que usa RUES), y permitirá en la futura Capa Silver/Gold hacer `JOIN` para saber, por ejemplo, a qué sector pertenece cada empresa matriculada.

## Estrategia de carga

Al ser un catálogo pequeño y prácticamente estático (499 filas), se hace una **carga completa por sobreescritura** (`overwrite`) en cada ejecución — no se necesita paginar ni hacer carga incremental (ese requisito ya lo cubre `03_Ingesta_Bronze_TRM.ipynb`).

## Regla de inmutabilidad (Capa Bronze)

* Se conservan los nombres originales de columna (`id`, `code`, `description`, `status`, `version`).
* No se hace *casting* manual de tipos.
* Solo se agregan `_ingested_at` y `_source` como columnas de auditoría.

In [ ]:
%python
import requests
import pandas as pd
from pyspark.sql import functions as F

BASE_URL = "https://www.datos.gov.co/resource/nuke-fusu.json"
LIMIT = 1000  # de sobra: el catálogo completo tiene ~499 filas
TABLA_DESTINO = "Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api"

print(f"Fuente: {BASE_URL}")
print(f"Destino: {TABLA_DESTINO}")

---

## Paso 1: Descargar el catálogo completo

In [ ]:
%python
def descargar_datos():
    registros = []
    offset = 0

    while True:
        params = {"$limit": LIMIT, "$offset": offset}
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        pagina = resp.json()

        if not pagina:
            break

        registros.extend(pagina)
        offset += LIMIT

    return registros


registros = descargar_datos()
df_pandas = pd.DataFrame(registros)
print(f"Catálogo CIIU descargado: {df_pandas.shape[0]} filas x {df_pandas.shape[1]} columnas")
df_pandas.head()

---

## Paso 2: Convertir a Spark DataFrame y agregar columnas de auditoría

In [ ]:
%python
# Sin casting manual: se respeta el tipo con el que Socrata entrega cada campo
df_spark = spark.createDataFrame(df_pandas)

df_bronze = (
    df_spark
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit(BASE_URL))
)

df_bronze.printSchema()
display(df_bronze.limit(10))

---

## Paso 3: Persistir en Delta Lake (Capa Bronze)

In [ ]:
%python
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_DESTINO)
)

print(f"Tabla Delta escrita: {TABLA_DESTINO}")

In [ ]:
%sql
DESCRIBE EXTENDED Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api;

In [ ]:
%sql
SELECT COUNT(*) AS total_filas FROM Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api;

SELECT * FROM Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api LIMIT 10;

---

## Ejemplo de uso futuro (Silver): descripción de la actividad económica principal de una empresa

No se ejecuta ni se persiste aquí — es solo un adelanto de cómo se cruzarán las tablas en la Capa Silver, sin modificar ninguna tabla Bronze.

In [ ]:
%sql
SELECT
  r.matricula,
  r.razon_social,
  r.cod_ciiu_act_econ_pri,
  c.description AS descripcion_actividad_principal
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api r
LEFT JOIN Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api c
  ON r.cod_ciiu_act_econ_pri = c.code
LIMIT 10;

---

## Siguiente paso

Continuar con [`05_Validaciones_Bronze.ipynb`](05_Validaciones_Bronze.ipynb) para el diagnóstico de calidad de las tres tablas.